# Audio Processing & Feature Extraction

This notebook covers:
1. **Feature Transforms**: MelSpectrogram, MFCC, Spectrogram, Delta features
2. **Audio Architectures**: 2D Convolutions over Mel-Spectrograms with ResNet backbones
3. **Custom Audio Models**: CNN-based audio classifiers

Perfect for audio classification, speech recognition, and sound event detection tasks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import numpy as np

# =====================================================================
# AUDIO FEATURE EXTRACTION TRANSFORMS
# =====================================================================

class AudioFeatureExtractor:
    """
    Complete audio feature extraction pipeline.
    Converts raw audio waveforms to spectral representations.
    """
    
    def __init__(self, sample_rate=16000, n_mels=128, n_mfcc=13, 
                 n_fft=400, hop_length=160):
        self.sample_rate = sample_rate
        self.n_mels = n_mels
        self.n_mfcc = n_mfcc
        self.n_fft = n_fft
        self.hop_length = hop_length
        
        # Initialize transforms
        self.spectrogram = T.Spectrogram(
            n_fft=n_fft,
            hop_length=hop_length,
            power=2.0
        )
        
        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_mels=n_mels,
            n_fft=n_fft,
            hop_length=hop_length,
            power=2.0
        )
        
        self.mfcc = T.MFCC(
            sample_rate=sample_rate,
            n_mfcc=n_mfcc,
            melkwargs={'n_fft': n_fft, 'hop_length': hop_length}
        )
    
    def extract_spectrogram(self, waveform):
        """
        Extract raw spectrogram from audio.
        Args:
            waveform: [1, time_steps] or [time_steps]
        Returns:
            spectrogram: [1, freq_bins, time_steps]
        """
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)
        
        spec = self.spectrogram(waveform)
        # Log scale
        spec = torch.log(spec + 1e-9)
        return spec
    
    def extract_mel_spectrogram(self, waveform):
        """
        Extract Mel spectrogram (frequency scale perceived by humans).
        Args:
            waveform: [1, time_steps] or [time_steps]
        Returns:
            mel_spec: [1, n_mels, time_steps]
        """
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)
        
        mel_spec = self.mel_spectrogram(waveform)
        # Log scale
        mel_spec = torch.log(mel_spec + 1e-9)
        return mel_spec
    
    def extract_mfcc(self, waveform):
        """
        Extract MFCC (Mel-Frequency Cepstral Coefficients).
        Better for speech recognition tasks.
        Args:
            waveform: [1, time_steps] or [time_steps]
        Returns:
            mfcc: [1, n_mfcc, time_steps]
        """
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)
        
        mfcc = self.mfcc(waveform)
        return mfcc
    
    def extract_delta(self, spectrogram):
        """
        Extract Delta (first-order time derivative) features.
        Captures temporal dynamics.
        Args:
            spectrogram: [1, freq_bins, time_steps]
        Returns:
            delta: [1, freq_bins, time_steps-1]
        """
        # Simple finite difference
        delta = spectrogram[:, :, 1:] - spectrogram[:, :, :-1]
        return delta
    
    def extract_delta_delta(self, spectrogram):
        """
        Extract Delta-Delta (acceleration/second-order derivative).
        Captures temporal acceleration.
        """
        delta = self.extract_delta(spectrogram)
        delta_delta = self.extract_delta(delta)
        return delta_delta
    
    def extract_all_features(self, waveform):
        """
        Extract all features at once.
        Returns:
            dict with all feature types
        """
        mel_spec = self.extract_mel_spectrogram(waveform)
        mfcc = self.extract_mfcc(waveform)
        delta = self.extract_delta(mel_spec)
        
        return {
            'mel_spectrogram': mel_spec,
            'mfcc': mfcc,
            'delta': delta,
            'spectrogram': self.extract_spectrogram(waveform)
        }


In [ ]:
# =====================================================================
# CNN-BASED AUDIO CLASSIFICATION
# =====================================================================

class AudioCNN(nn.Module):
    """
    CNN for audio classification using Mel-Spectrogram images.
    Treats spectrograms as 2D images and applies 2D convolutions.
    """
    
    def __init__(self, n_mels=128, n_classes=10, dropout=0.5):
        super().__init__()
        
        # Conv blocks: treat spectrogram as 2D image
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2)
        )
        
        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        # Classification head
        self.fc = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, mel_spectrogram):
        """
        Args:
            mel_spectrogram: [batch_size, 1, n_mels, time_steps]
        Returns:
            logits: [batch_size, n_classes]
        """
        x = self.conv1(mel_spectrogram)
        x = self.conv2(x)
        x = self.conv3(x)
        
        x = self.gap(x)  # [batch, 128, 1, 1]
        x = x.view(x.size(0), -1)  # [batch, 128]
        
        logits = self.fc(x)  # [batch, n_classes]
        return logits


class AudioResNet(nn.Module):
    """
    ResNet-style architecture for audio.
    Residual connections allow deeper networks.
    """
    
    def __init__(self, n_classes=10, depth=18):
        super().__init__()
        
        # Initial conv layer
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # Residual blocks
        if depth == 18:
            layers = [2, 2, 2, 2]
        elif depth == 34:
            layers = [3, 4, 6, 3]
        
        self.layer1 = self._make_layer(64, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(64, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(128, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(256, 512, layers[3], stride=2)
        
        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Classification head
        self.fc = nn.Linear(512, n_classes)
    
    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        layers = []
        # Downsample if needed
        if stride != 1 or in_channels != out_channels:
            layers.append(nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            ))
        
        # Residual blocks
        for _ in range(blocks):
            layers.append(self._residual_block(out_channels, out_channels))
        
        return nn.Sequential(*layers)
    
    def _residual_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        
        return x


In [ ]:
# =====================================================================
# AUDIO + RNN HYBRID ARCHITECTURE
# =====================================================================

class AudioRNNClassifier(nn.Module):
    """
    Hybrid architecture combining CNN feature extraction with RNN temporal modeling.
    CNN extracts spectral features, RNN captures temporal dependencies.
    """
    
    def __init__(self, n_mels=128, n_classes=10, hidden_dim=256, n_layers=2, dropout=0.5):
        super().__init__()
        
        # CNN feature extractor
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3, 3), padding=(1, 1)),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d((2, 2)),
            
            nn.Conv2d(32, 64, kernel_size=(3, 3), padding=(1, 1)),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d((2, 2)),
            
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=(1, 1)),
            nn.ReLU(),
            nn.BatchNorm2d(128)
        )
        
        # After pooling: n_mels//4, time_steps//4
        cnn_out_dim = (n_mels // 4) * 128
        
        # RNN for temporal modeling
        self.lstm = nn.LSTM(
            input_size=cnn_out_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        # Classification head
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, mel_spectrogram):
        """
        Args:
            mel_spectrogram: [batch_size, 1, n_mels, time_steps]
        Returns:
            logits: [batch_size, n_classes]
        """
        # CNN feature extraction
        x = self.cnn(mel_spectrogram)  # [batch, 128, n_mels//4, time_steps//4]
        
        # Reshape for RNN: [batch, time_steps//4, (n_mels//4)*128]
        batch_size = x.size(0)
        x = x.permute(0, 3, 1, 2)  # [batch, time_steps//4, 128, n_mels//4]
        x = x.reshape(batch_size, x.size(1), -1)  # [batch, time_steps//4, 128*(n_mels//4)]
        
        # LSTM temporal modeling
        _, (h_n, c_n) = self.lstm(x)
        
        # Use final hidden state from both directions
        h_final = torch.cat([h_n[-1], h_n[-2]], dim=1)  # [batch, hidden_dim*2]
        
        # Classification
        logits = self.fc(h_final)
        return logits


In [ ]:
# =====================================================================
# DEMONSTRATION AND TESTING
# =====================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")
    
    # Test 1: Feature Extraction
    print("=" * 60)
    print("AUDIO FEATURE EXTRACTION")
    print("=" * 60)
    
    extractor = AudioFeatureExtractor(sample_rate=16000, n_mels=128, n_mfcc=13)
    
    # Create mock audio waveform (1 second at 16kHz)
    waveform = torch.randn(1, 16000)
    
    mel_spec = extractor.extract_mel_spectrogram(waveform)
    mfcc = extractor.extract_mfcc(waveform)
    delta = extractor.extract_delta(mel_spec)
    
    print(f"Waveform shape: {waveform.shape}")
    print(f"Mel-Spectrogram shape: {mel_spec.shape}")
    print(f"MFCC shape: {mfcc.shape}")
    print(f"Delta shape: {delta.shape}")
    print(f"✓ Feature extraction working!\n")
    
    # Test 2: Audio CNN
    print("=" * 60)
    print("AUDIO CNN CLASSIFIER")
    print("=" * 60)
    
    audio_cnn = AudioCNN(n_mels=128, n_classes=10).to(device)
    
    # Mock mel-spectrogram batch
    mel_batch = torch.randn(4, 1, 128, 100).to(device)
    
    audio_cnn.eval()
    with torch.no_grad():
        logits = audio_cnn(mel_batch)
    
    print(f"Input shape: {mel_batch.shape}")
    print(f"Output shape: {logits.shape}")
    print(f"✓ Audio CNN working!\n")
    
    # Test 3: Audio ResNet
    print("=" * 60)
    print("AUDIO RESNET")
    print("=" * 60)
    
    audio_resnet = AudioResNet(n_classes=10, depth=18).to(device)
    
    audio_resnet.eval()
    with torch.no_grad():
        logits = audio_resnet(mel_batch)
    
    print(f"ResNet output shape: {logits.shape}")
    print(f"✓ Audio ResNet working!\n")
    
    # Test 4: Audio-RNN Hybrid
    print("=" * 60)
    print("AUDIO-RNN HYBRID CLASSIFIER")
    print("=" * 60)
    
    audio_rnn = AudioRNNClassifier(n_mels=128, n_classes=10, hidden_dim=256).to(device)
    
    audio_rnn.eval()
    with torch.no_grad():
        logits = audio_rnn(mel_batch)
    
    print(f"Hybrid output shape: {logits.shape}")
    print(f"✓ Audio-RNN hybrid working!\n")
    
    print("✅ All audio models working correctly!")
